# Gold — dim_channel_scd2 (SCD 2, poc `scd_type_2_new` method)

One row per `channel_id` × version. Built like poc `dim_family`: read stg → select columns → metadata → `add_scd_checksum` + `add_checksums` → `scd_type_2(df, table)` (also in **full** mode, like the poc new helper: full only drops the date filter).

**SCD columns (major → new version):** `channel_title`, `custom_url`, `country`: the channel's identity, what a report by channel should show "as it was". Everything else (description, thumbnail, language) is **minor**: updated in place on the current row.

Version dates: first version `start_date = created_at`; a major change closes the old row with `end_date = modified_at` and opens the new one with `start_date = modified_at`; current row `end_date = 9999-12-31 23:59:59`, `current_flag = 'Y'`.

Fact join "as of": `f.channel_id = d.channel_id AND f.ts >= d.start_date AND f.ts < d.end_date`, or `current_flag = 'Y'` for today's view. Not in the daily job yet; the live `dim_channel` is unchanged.

In [ ]:
%run ./_scd

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = today
SILVER, GOLD = dbutils.widgets.get("silver"), dbutils.widgets.get("gold")
LOAD_TYPE = dbutils.widgets.get("load_type")
FROM_DATE, TO_DATE = default_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))
PRIMARY_KEY = ["channel_id"]
print(f"full: every read of {SILVER}.stg_channels -> {GOLD}" if LOAD_TYPE == "full"
      else f"incremental: {SILVER}.stg_channels snapshot_date in [{FROM_DATE}, {TO_DATE}) -> {GOLD}")
SCD_COLUMNS = ["channel_title", "custom_url", "country"]

## Source → columns

| Column | Source (`stg_channels`) | Note |
|---|---|---|
| `channel_id` | `channel_id` | primary key |
| `channel_title` ... `published_at` | same name | descriptive attributes |
| `created_at` | `MIN(created_at)` over every read of the channel | first time the app saw the channel |
| `modified_at` | `snapshot_ts` of the last read in the window | when these values were read |
| `etl_date`, `etl_by` | job time, `Databricks` | metadata |

`created_at` / `modified_at` are **not** hashed (passed as `exclude`), so a re-read with the same values writes nothing and `etl_date` keeps meaning "changed on".

In [ ]:
first_seen = spark.table(f"{SILVER}.stg_channels").groupBy("channel_id").agg(F.min("created_at").alias("created_at"))
src = (stg_window(f"{SILVER}.stg_channels", LOAD_TYPE, FROM_DATE, TO_DATE).drop("created_at")
       .join(first_seen, "channel_id")
       .select("channel_id", "channel_title", "channel_description", "custom_url", "country", "default_language",
               "thumbnail_url", "published_at", "created_at", F.col("snapshot_ts").alias("modified_at"))
       .withColumn("etl_date", F.current_timestamp())
       .withColumn("etl_by", F.lit("Databricks")))

src = add_scd_checksum(src, SCD_COLUMNS)
src = add_checksums(src, PRIMARY_KEY, exclude=["created_at", "modified_at"])
print(f"source rows: {src.count()}")

In [ ]:
TABLE = f"{GOLD}.dim_channel_scd2"
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE} (
        id BIGINT GENERATED BY DEFAULT AS IDENTITY,
        channel_id STRING, channel_title STRING, channel_description STRING, custom_url STRING,
        country STRING, default_language STRING, thumbnail_url STRING, published_at TIMESTAMP,
        created_at TIMESTAMP, modified_at TIMESTAMP, etl_date TIMESTAMP, etl_by STRING,
        primary_checksum STRING, record_checksum STRING, scd_checksum STRING,
        current_flag STRING, start_date TIMESTAMP, end_date TIMESTAMP
    ) USING DELTA
    COMMENT 'Channel dimension, SCD 2 (poc scd_type_2_new method). id = surrogate key per version, -1 = Unknown.'
""")
counts = scd_type_2(src, TABLE)
print(f"minor (update in place): {counts['minor']} · major (new version): {counts['major']} · new channels: {counts['new']}")

spark.sql(f"""
    INSERT INTO {TABLE} (id, channel_title, etl_date, etl_by, current_flag, start_date, end_date)
    SELECT -1, 'Unknown', current_timestamp(), 'Databricks', 'Y', TIMESTAMP '1900-01-01', TIMESTAMP '{MAX_DATE}'
    WHERE NOT EXISTS (SELECT 1 FROM {TABLE} WHERE id = -1)
""")

In [ ]:
c = spark.sql(f"""
    SELECT COUNT(*) AS n,
           SUM(CASE WHEN current_flag = 'Y' THEN 1 ELSE 0 END) AS cur,
           COUNT(DISTINCT CASE WHEN current_flag = 'Y' THEN channel_id END) AS cur_ids,
           SUM(CASE WHEN start_date > end_date THEN 1 ELSE 0 END) AS bad_dates
    FROM {TABLE} WHERE channel_id IS NOT NULL
""").first()
assert c["cur"] == c["cur_ids"], f"{TABLE}: a channel has more than one current row"
assert c["bad_dates"] == 0, f"{TABLE}: {c['bad_dates']} rows with start_date > end_date"
print(f"✅ {TABLE}: {c['n']} versions, {c['cur']} current channels")